# Metal3D Prediction

![Metal3D Prediction](https://proto-bio.github.io/proto-assets/images/tool/metal3d/hero.png)

This notebook demonstrates `run_metal3d_prediction`, which predicts zinc-ion binding sites in a protein structure. It scores candidate metal-coordinating residues, returns clustered metal-site coordinates with a per-site confidence, and appends the top predicted zinc to an annotated structure.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("metal3d")
display_overview("metal3d")
display_docs_section("metal3d", "Background")

## Available tools

In [ ]:
display_available_tools("metal3d")

### `run_metal3d_prediction`

Metal3D scores candidate metal-coordinating residues and predicts where zinc ions bind. Pass `candidate_residues` to restrict scoring to a known pocket and obtain per-residue probabilities, or omit it to evaluate all metal-binding residue types across the protein. The default `metal3d-original` checkpoint is the published Metal3D model; `metal3d-cat` and `metal3d-clean` are dEVA's retrained variants.

In [ ]:
from pathlib import Path

from proto_tools.tools.structure_scoring.metal3d import (
    Metal3DPredictionConfig,
    Metal3DPredictionInput,
    Metal3DStructureInput,
    run_metal3d_prediction,
)
from proto_tools.entities.structures import Structure

In [ ]:
# Display input docs
display_api_reference("metal3d", "input", "run_metal3d_prediction")

# Carbonic anhydrase (2VVB) — score the His94/His96/His119 zinc-coordinating triad
structure = Structure(structure=str(Path("..") / "example_input_fixture.pdb"))

inputs = Metal3DPredictionInput(
    inputs=[Metal3DStructureInput(structure=structure, candidate_residues={"X": [94, 96, 119]})]
)

In [ ]:
# Display config docs
display_api_reference("metal3d", "config", "run_metal3d_prediction")

# Configure prediction | see docs above for all fields
config = Metal3DPredictionConfig(
    model_checkpoint="metal3d-original",
    probability_threshold=0.2,
    device="cuda",
)

In [ ]:
# Run the prediction function
result = run_metal3d_prediction(inputs, config)

In [ ]:
# Display output docs
display_api_reference("metal3d", "output", "run_metal3d_prediction")

# Print the top predicted zinc site and per-residue probabilities
prediction = result.results[0]
print(f"Site found: {prediction.found} | p(metal): {prediction['pmetal']:.3f}")
for site in prediction.sites:
    print(f"  Zn site: ({site.x:.2f}, {site.y:.2f}, {site.z:.2f})  p={site['probability']:.3f}")
for rp in prediction.residue_probabilities:
    print(f"  {rp.residue_name}{rp.residue_id} (chain {rp.chain_id}): p={rp['probability']:.3f}")

## Export Results

Predictions can be exported to JSON, or the annotated structures (with the predicted zinc appended) to PDB.

In [ ]:
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="metal3d_sites", export_path=output_dir, file_format="json")
print(f"Predictions exported to {output_dir / 'metal3d_sites.json'}")

result.export(name="metal3d_annotated", export_path=output_dir, file_format="pdb")
print(f"Annotated structures exported to {output_dir / 'metal3d_annotated/'}")